In [ ]:
%load ../lib/RecursiveSet.jsh
%load Formula.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

In [ ]:
<T> T first(RecursiveSet<T> S) {
    return S.iterator().next();
}

The function `range` takes two natural numbers `start`and `stop`as arguments.  It
returns a set containing the numbers from `start` up to and including the number `stop`. 

In [ ]:
RecursiveSet<Integer> range(int start, int stop) {
    RecursiveSet<Integer> result = new RecursiveSet<>();
    for (int i = start; i <= stop; ++i) {
        result.add(i);
    }
    return result;
}

In [ ]:
range(1, 8)

# The Eight Queens Puzzle

The <a href="https://en.wikipedia.org/wiki/Eight_queens_puzzle">eight queens puzzle</a> is the problem of placing eight chess queens on a chessboard so that no two queens can attack each other.  In <a href="https://en.wikipedia.org/wiki/Chess">chess</a> a queen can attack another piece if this piece is either
<ol>
    <li>in the same row,</li>
    <li>in the same column, or</li>
    <li>in the same diagonal.</li>
</ol>
The image below shows a queen in row 3, column 4.  All the locations where a piece can be captured by this queen are marked with an arrow.

<img src="queen-captures.png">

We will solve this puzzle by coding it as a formula of propositional logic.  This formula will be solvable iff the eight queens puzzle has a solution.  We will use the algorithm of *Davis and Putnam* to compute the solution of this formula.

We load the implementation of the Davis-Putnam algorithm that uses the Jeroslow-Wang heuristic.  It provides the function `solve`.

In [ ]:
%load 07-Davis-Putnam-JW.jsh

The function $\texttt{var}(r, c)$ takes a row $r$ and a column $c$ and returns the string $\texttt{'Q<}r\texttt{,}c\texttt{>'}$.  This string is interpreted as a propositional variable specifying that there is a queen in row $r$ and column $c$.  The image below shows how theses variables correspond to the positions on a chess board.

<img src="queens-vars.png">

The function `varName(row, col)` takes two integers `row` and `col` as its argument and returns a string of the form `Q<row,col>`.
This string is interpreted as a propositional variable.  This variable is `true` if there is a queen in the given row and column on the board.

In [ ]:
String varName(int row, int col) {
    return "Q<" + row + "," + col + ">";
}

In [ ]:
varName(11, 3)

Given a set of propositional variables $S$, the function $\texttt{atMostOne}(S)$ returns a set containing clauses that expresses the fact that **at most one** of the variables in $S$ is `True`:
$$ \texttt{atMostOne}(S) = \bigl\{ \{\neg p, \neg q\} \bigm| \langle p, q\rangle \in S \times S \wedge p \not= q\bigr\} $$

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atMostOne(RecursiveSet<String> S) {
    return flatMap(S, p -> S.filterMap(q -> !p.equals(q),
                                       q -> set(new Neg(p), new Neg(q))));
}

Note that strings have to be compared with the method `equals`, since the operator `!=` would compare the references of the strings.

In [ ]:
atMostOne(set("a", "b", "c"))

Given a `row` and the size of the board `n`, the procedure `atMostOneInRow(row, n)` computes a set of clauses that is `True` if and only there is at most one queen in the
given row.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atMostOneInRow(int row, int n) {
    return atMostOne(range(1, n).map(col -> varName(row, col)));
}

In [ ]:
range(1, 4).map(col -> varName(2, col))

In [ ]:
atMostOneInRow(2, 4)

Given a column `col` and the size of the board `n`, the procedure `oneInColumn(col, n)` computes a set of clauses that is true if and only if there is at least one queen in the given column.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> oneInColumn(int col, int n) {
    return set(range(1, n).map(row -> new Var(varName(row, col))));
}

In [ ]:
oneInColumn(2, 4)

Given a number `k` and the size of the board `n`, the procedure `atMostOneInFallingDiagonal(k, n)` computes a set of clauses that is `True` if and only if there 
is at most one queen in the falling diagonal specified by the equation
```
     row - col = k.
```

The cartesian product `Numbers × Numbers` is a set of objects of the record class `Pair`.  The components of a pair are accessed via the methods `first()` and `second()`.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atMostOneInFallingDiagonal(int k, int n) {
    var Numbers = range(1, n);
    return atMostOne(Numbers.cartesianProduct(Numbers)
                            .filterMap(pair -> pair.first() - pair.second() == k,
                                       pair -> varName(pair.first(), pair.second())));
}

In [ ]:
atMostOneInFallingDiagonal(0, 4)

Given a number `k` and the size of the board `n`, the procedure `atMostOneInRisingDiagonal(k, n)` computes a set of clauses that is `True` 
if and only if there is at most one queen in the rising diagonal specified by the equation
```
    row + col = k. 
```

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atMostOneInRisingDiagonal(int k, int n) {
    var Numbers = range(1, n);
    return atMostOne(Numbers.cartesianProduct(Numbers)
                            .filterMap(pair -> pair.first() + pair.second() == k,
                                       pair -> varName(pair.first(), pair.second())));
}

In [ ]:
atMostOneInRisingDiagonal(5, 4)

The function `allClauses(n)` takes the size of the board $n$ and computes a set of clauses that specify that
* there is at most one queen in every row,
* there is at most one queen in every rising diagonal,
* there is at most one queen in every falling diagonal, and
* there is at least one queen in every column.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> allClauses(int n) {
    return flatMap(range(   1,       n), row -> atMostOneInRow(row, n))
          .union(flatMap(range(   3, 2*n - 1), k   -> atMostOneInRisingDiagonal(k, n)))
          .union(flatMap(range(-n + 2, n - 2), k   -> atMostOneInFallingDiagonal(k, n)))
          .union(flatMap(range(   1,       n), col -> oneInColumn(col, n)));
}

In [ ]:
var clauses = allClauses(8);
for (var clause : clauses) {
    System.out.println(clause);
}

For the 8 queens problem, the set of all clauses contains 512 clauses.  There are 64 variables.

In [ ]:
clauses.size()

The function $\texttt{queens}(n)$ solves the *n queens problem*.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> queens(int n) {
    var Clauses     = allClauses(n);
    var Solution    = solve(Clauses);
    RecursiveSet<Literal> EmptyClause = set();
    if (Solution.contains(EmptyClause)) {
        System.out.println("The problem is not solvable for " + n + " queens!");
    }
    return Solution;
}

The *8 queens problem* can be solved in less than a tenth of a second using the
pure *Davis Putnam algorithm* that does not use the Jereslow-Wang heuristic. If we want to solve the *16 queens problem*, we need to use the Jereslow-Wang heuristic.  With this heuristic, the *16 queens problem* is then solved in roughly 3 seconds.

In [ ]:
long start = System.currentTimeMillis();
var Solution = queens(16);
System.out.println("queens: " + (System.currentTimeMillis() - start) + " ms");

var Queens = Solution.filterMap(u -> first(u) instanceof Var, u -> first(u));
System.out.println(Queens);

# Visualization

The function `removeNegativeLiterals` takes one input:
* `Solution` is a set of unit clauses.

The function returns the set of all those unit clauses in `Solution` that do not contain negative literals.

In the function below, the condition `lit instanceof Var(var p)` checks whether the literal `lit` is a variable.  If this is the case, the name of this variable is bound to `p`.

In [ ]:
RecursiveSet<String> removeNegativeLiterals(RecursiveSet<RecursiveSet<Literal>> Solution) {
    RecursiveSet<String> Result = set();
    for (var clause : Solution) {
        for (var lit : clause) {
            if (lit instanceof Var(var p)) {
                Result.add(p);
            }
        }
    }
    return Result;
}

In [ ]:
RecursiveSet<RecursiveSet<Literal>> inputSet = set(
    set(new Var("p")),
    set(new Neg("q")),
    set(new Neg("r")),
    set(new Var("s"))
);
removeNegativeLiterals(inputSet)

The function `extractRowCol` takes one argument:
* `varName` is a string of the form `Q<row,col>`.

It returns the pair `(row, col)`.

In [ ]:
Pair<Integer, Integer> extractRowCol(String varName) {
    int left  = varName.indexOf('<');
    int comma = varName.indexOf(',');
    int right = varName.indexOf('>');
    int row   = Integer.parseInt(varName.substring(left + 1, comma));
    int col   = Integer.parseInt(varName.substring(comma + 1, right));
    return new Pair<>(row, col);
}

In [ ]:
extractRowCol("Q<13,9>")

The function `transform` takes a solution of the n queens problem and returns a map that maps every row to the column of the queen in this row.

In [ ]:
Map<Integer, Integer> transform(RecursiveSet<RecursiveSet<Literal>> Solution) {
    Map<Integer, Integer> Result = new TreeMap<>();
    for (var name : removeNegativeLiterals(Solution)) {
        var rowCol = extractRowCol(name);
        Result.put(rowCol.first(), rowCol.second());
    }
    return Result;
}

In [ ]:
transform(Solution)

The function `showSolution(Solution, width)` takes a set of unit clauses that represents a solution to the n queens puzzle. It displays this solution on a chess board.

* `Solution`: A `RecursiveSet` containing the variable assignment (as unit clauses).
  The function transforms this set into a map from row indices to column indices.
  If `transformed.get(row) == col`, then the queen in `row` is placed in `col`.
* `width`: Specifies the size of the board as a percentage of the width of notebook.

The HTML code is assembled using *text blocks*, i.e. multi-line strings that are enclosed in `"""`.  The method `formatted` replaces the placeholders `%s` and `%d` in a string by its arguments.  The function `display`, which is provided by the IJava kernel, displays a string as HTML.

As Java does not support default values for parameters, we define a second function `showSolution` with just one parameter that uses the width `50%`.

In [ ]:
void showSolution(RecursiveSet<RecursiveSet<Literal>> Solution, String width) {
    var transformed = transform(Solution);
    int n           = transformed.size();
    var html        = new StringBuilder();
    html.append("""
        <div style="display:grid; grid-template-columns:repeat(%d, 1fr); width:%s;
                    aspect-ratio: 1/1; border: 2px solid black;">
        """.formatted(n, width));
    for (int row = 1; row <= n; row++) {
        for (int col = 1; col <= n; col++) {
            String bgColor     = (row + col) % 2 == 0 ? "#f0d9b5" : "#b58863";
            String cellContent = "";
            if (transformed.containsKey(row) && transformed.get(row) == col) {
                cellContent = """
                    <svg viewBox="0 0 100 100" style="width: 80%; height: 80%; display: block;">
                        <text x="50%" y="55%" font-size="90" text-anchor="middle"
                              dominant-baseline="middle" fill="black">♕</text>
                    </svg>""";
            }
            html.append("""
                <div style="display: flex; align-items: center; justify-content: center;
                            background-color:%s; overflow: hidden;">%s</div>
                """.formatted(bgColor, cellContent));
        }
    }
    html.append("</div>");
    display(html.toString(), "text/html");
}

void showSolution(RecursiveSet<RecursiveSet<Literal>> Solution) {
    showSolution(Solution, "50%");
}

In [ ]:
showSolution(Solution, "70%");